In [ ]:
install.packages('phers')

In [ ]:
#load packages
library(data.table)
library(bigrquery)
library(phers)
library(dplyr)

In [ ]:
#define variables
# CDR <- system("wb resolve --id=C2025Q4R6", intern = TRUE)
# GOOGLE_PROJECT <- system("gcloud config get-value project", intern = TRUE)
# WORKSPACE_BUCKET <- "gs://phers-wb-quick-tomato-9709"

CDR <- 'wb-silky-artichoke-2408.C2025Q4R6'
GOOGLE_PROJECT <- 'wb-quick-tomato-9709'
WORKSPACE_BUCKET <- 'gs://phers-wb-quick-tomato-9709'

CDR
GOOGLE_PROJECT
WORKSPACE_BUCKET

In [ ]:
#pull cohort from biobank and produce icdOccurrences data table
icdOcc_sql <- glue::glue("
  SELECT DISTINCT person_id, 
  CAST(condition_source_value AS STRING) AS icd,
  vocabulary_id AS flag
  FROM `{CDR}.condition_occurrence`AS co
  JOIN `{CDR}.concept` AS c
  ON
    co.condition_source_concept_id = c.concept_id
  WHERE condition_source_value IS NOT NULL
      AND 
        c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
  LIMIT 500000
")
icdOccurrences <- bq_table_download(bq_project_query(
  Sys.getenv('GOOGLE_PROJECT'), icdOcc_sql))
setDT(icdOccurrences)

icdOccurrences[, flag := fifelse(flag == "ICD9CM", 9L, 10L)]

In [ ]:
icdOccurrences

In [ ]:
#map icd occurrences to phecodes
phecodeOccurrences <- getPhecodeOccurrences(icdOccurrences)

In [ ]:
phecodeOccurrences

In [ ]:
#map weights to phecodes and people using preCalcWeights
weights = left_join(phecodeOccurrences, preCalcWeights, by = 'phecode')
weights <- weights[ ,pred := NULL]

In [ ]:
weights

In [ ]:
#save disease to phecode mapping
diseasePhecodeMap <- mapDiseaseToPhecode()

In [ ]:
diseasePhecodeMap

In [ ]:
#generate risk scores
pheRS <- getScores(weights, diseasePhecodeMap)
pheRS